In [ ]:
!python --version

In [ ]:
!pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import scipy as sp
from sklearn.metrics import rand_score
%matplotlib inline

n_clusters = 3

## EDA


In [ ]:
df = pd.read_csv("spiral-dataset.csv", sep="\t", header=None, names=["x", "y", "class"])
df.head()

In [ ]:
df["class"].unique()

## 1.2.3.Generate a figure from the given dataset that resembles Figure 1.


In [ ]:
df.plot.scatter(x="x", y="y", c="class", colormap='viridis')

### Implement the k-means clustering algorithm. And do the following:

### • 2.a) Run your k-means algorithm on the given dataset setting the value k=3 (because visually we only have 3 clusters to worry about). And do not forget to randomly initialize the 3 centroids.


In [ ]:
# function to generate 3 random centroids within the bounds of the data
def random_centroids():
    # find the bounds of the data (doesn't need to be normalized per the instructions)
    max_x = df["x"].max()
    max_y = df["y"].max()

    # generate `n_clusters` random centroids within the bounds of the data
    centroids = [(np.random.rand() * max_x, np.random.rand() * max_y) for _ in range(n_clusters)]

    # convert to a numpy array and return
    return np.array(centroids)



# function to find the closest centroid for each data point
def find_classes(df, centroids):
    # this is going to be an array of the closest centroid for each data point
    closest_centroids = []

    # build the array of closest centroids
    for index, row in df.iterrows():
        # get an array of the distances from the data point to each centroid
        distances = np.array([sp.spatial.distance.euclidean((row["x"], row["y"]), centroid) for centroid in centroids])
        # find the index of the closest centroid and add it to the list
        best_centroid = np.argmin(distances)
        closest_centroids.append(best_centroid)

    return np.array(closest_centroids)

Now we're going to make a function to iterate:

1. Attach a kmeans_cluster value to each row
2. Use the new dataframe with the kmeans_cluster column to find the mean point for the cluster.
3. Move the centroid to the mean point of each cluster.
4. Repeat until it converges


In [ ]:
def do_kmeans(input_df, debug=False):

    # make a copy of the input dataframe to work with
    kmeans_df = input_df.copy()

    # initialize the centroids and assign the initial clusters
    centroids = random_centroids()
    if debug:
        print("Initial centroids:")
        print(f"\t({centroids[0][0]:.2f}, {centroids[0][1]:.2f})")
        print(f"\t({centroids[1][0]:.2f}, {centroids[1][1]:.2f})")
        print(f"\t({centroids[2][0]:.2f}, {centroids[2][1]:.2f})")

    clusters = find_classes(kmeans_df, centroids)

    # initialize some variables to prepare the preconditions for the loop
    old_centroids = [
        (0,0),
        (0,0),
        (0,0)
    ]
    kmeans_df["kmeans_cluster"] = clusters

    # ITERATION: loop until the centroids don't move
    while not np.array_equal(centroids, old_centroids):

        # split the dataframe into three clusters based on the kmeans_cluster column
        cluster1 = kmeans_df[kmeans_df["kmeans_cluster"] == 0]
        cluster2 = kmeans_df[kmeans_df["kmeans_cluster"] == 1]
        cluster3 = kmeans_df[kmeans_df["kmeans_cluster"] == 2]

        # store the old centroids before updating them
        old_centroids = centroids.copy()

        # update the centroids to be the mean of the points in each cluster
        # if a cluster is empty, don't move the centroid
        centroids[0] = (cluster1["x"].mean(), cluster1["y"].mean()) if len(cluster1) > 0 else centroids[0]
        centroids[1] = (cluster2["x"].mean(), cluster2["y"].mean()) if len(cluster2) > 0 else centroids[1]
        centroids[2] = (cluster3["x"].mean(), cluster3["y"].mean()) if len(cluster3) > 0 else centroids[2]

        # print the centroid movement
        if debug:
            if not np.array_equal(centroids[0], old_centroids[0]):
                print(f"Centroid 1 moved from ({old_centroids[0][0]:.2f}, {old_centroids[0][1]:.2f}) to ({centroids[0][0]:.2f}, {centroids[0][1]:.2f})")
            if not np.array_equal(centroids[1], old_centroids[1]):
                print(f"Centroid 2 moved from ({old_centroids[1][0]:.2f}, {old_centroids[1][1]:.2f}) to ({centroids[1][0]:.2f}, {centroids[1][1]:.2f})")
            if not np.array_equal(centroids[2], old_centroids[2]):
                print(f"Centroid 3 moved from ({old_centroids[2][0]:.2f}, {old_centroids[2][1]:.2f}) to ({centroids[2][0]:.2f}, {centroids[2][1]:.2f})")

        # reassign the clusters based on the new centroids
        kmeans_df["kmeans_cluster"] = find_classes(kmeans_df, centroids)
    # END ITERATION
    
    if debug:
        print("Final centroids:")
        print(f"\t({centroids[0][0]:.2f}, {centroids[0][1]:.2f})")
        print(f"\t({centroids[1][0]:.2f}, {centroids[1][1]:.2f})")
        print(f"\t({centroids[2][0]:.2f}, {centroids[2][1]:.2f})")

    # we need to add 1 to the kmeans_cluster column to match the original class labels
    kmeans_df["kmeans_cluster"] = kmeans_df["kmeans_cluster"] + 1

    # return the dataframe with the kmeans_cluster column and the final centroids to be used for the SSE calculation
    return kmeans_df, centroids

In [ ]:
kmeans_df, centroids = do_kmeans(df, debug=True)

## • 2.b) Once your k-means algorithm has converged above, stop and from your clustering result compute the intrinsic performance metric: Sum of Squared Error, SSE (smaller the better), and the extrinsic performance metric: Rand-Index, RI (higher the better).


Let's find SSE first


In [ ]:
def kmeans_sse(df, centroids):
    # our running total
    total = 0

    # iterate through each row
    for index, row in df.iterrows():
        # select the centroid for the cluster that the row belongs to based on the column we added to the dataframe
        centroid = centroids[int(row["kmeans_cluster"] - 1)]
        # the error is the distance from the data point to the centroid
        error = sp.spatial.distance.euclidean((row["x"], row["y"]), centroid)
        # add the squared error to the total
        total += error ** 2

    return total

In [ ]:
sse = kmeans_sse(kmeans_df, centroids)
print(f"SSE: {sse}")
ri = rand_score(kmeans_df["class"], kmeans_df["kmeans_cluster"])
print(f"Rand Index: {ri}")

## • 2.c) Repeat Task (2.a) & (2.b) another 9 (nine) times randomizing again the initial centroids,and report out of the 10 runs of k-means what is the best SSE & RI you could get.


In [ ]:
# some lists to hold our results, initialize with the results from the first attempt
kmeans_df_list = [kmeans_df]
centroids_list = [centroids]
sse_list = [sse]
ri_list = [ri]

# re-print this because I have OCD
print(f"Attempt 01 -> SSE: {sse:.10f}, Rand Index: {ri:.10f}")

# loop 9 times
for class_value in range(9):
    # get the results for this attempt
    this_df, this_centroids = do_kmeans(df)
    this_sse = kmeans_sse(this_df, this_centroids)
    this_ri = rand_score(this_df["class"], this_df["kmeans_cluster"])

    # print the results for this attempt
    print(f"Attempt {class_value+2:02d} -> SSE: {this_sse:.10f}, Rand Index: {this_ri:.10f}")
    
    # append the results to the lists
    kmeans_df_list.append(this_df)
    centroids_list.append(this_centroids)
    sse_list.append(this_sse)
    ri_list.append(this_ri)

# find the best one
best_sse_i = np.argmin(sse_list)
best_ri_i = np.argmax(ri_list)
print(f"\nBest SSE:        Attempt {best_sse_i+1:02d} with SSE:     {sse_list[best_sse_i]:.10f}")
print(f"Best Rand Index: Attempt {best_ri_i+1:02d} with Rand Index: {ri_list[best_ri_i]:.10f}")

# sanity check
if best_sse_i != best_ri_i:
    print("What??? SSE and Rand Index don't agree on the best attempt?")

## • 2.d) Please draw the clustering results (like Figure 1).


In [ ]:
kmeans_df_list[best_sse_i].plot.scatter(x="x", y="y", c="kmeans_cluster", colormap='viridis', title=f"Best SSE model (Attempt {best_sse_i+1:02d})")

In [ ]:
kmeans_df_list[best_ri_i].plot.scatter(x="x", y="y", c="kmeans_cluster", colormap='viridis', title=f"Best RI model (Attempt {best_ri_i+1:02d})")

-
-
-
-
-
-
-
-


## (40 pts) Implement the Hierarchical clustering algorithm. And do the following:
## • 3.a) Using the “single linkage” method, run the hierarchical clustering algorithm on the dataset, and get a 3-cluster result (by cutting the dendrogram at a certain height), and report SSE and RI.

The cell below is my implementation of pairwise distance.  I have deactivated it in favor of scipy's `pdist()` function

In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import scipy as sp
from sklearn.metrics import rand_score
%matplotlib inline

Make the point-distance matrix

In [13]:
# reload the data
df = pd.read_csv("spiral-dataset.csv", sep="\t", header=None, names=["x", "y", "class"])

# pdist() is so much easier...
distance_matrix = sp.spatial.distance.pdist(df[["x", "y"]])
distance_matrix = sp.spatial.distance.squareform(distance_matrix)

# EDA
print(f"Distance matrix shape: {distance_matrix.shape}")
print(distance_matrix[:5, :5])
# we're going to ignore the diaganol and lower triangle since those are duplicate values. fill them in with infinity so they are ignored in argmin()

# remove the diagonal and lower triangle by setting them to infinity so they are ignored in argmin()
print("\nDistance matrix with diagonal and lower triangle set to infinity:")
distance_matrix[np.tril_indices_from(distance_matrix)] = np.inf
print(distance_matrix[:5, :5])

Distance matrix shape: (312, 312)
[[0.         1.03077641 1.98494332 2.97741499 3.88104367]
 [1.03077641 0.         0.95524866 1.94743421 2.85043856]
 [1.98494332 0.95524866 0.         0.99247166 1.90065778]
 [2.97741499 1.94743421 0.99247166 0.         0.91787799]
 [3.88104367 2.85043856 1.90065778 0.91787799 0.        ]]

Distance matrix with diagonal and lower triangle set to infinity:
[[       inf 1.03077641 1.98494332 2.97741499 3.88104367]
 [       inf        inf 0.95524866 1.94743421 2.85043856]
 [       inf        inf        inf 0.99247166 1.90065778]
 [       inf        inf        inf        inf 0.91787799]
 [       inf        inf        inf        inf        inf]]


The single linkage algorithm

In [14]:
def single_linkage(df, distance_matrix):

    # where we'll save the class values
    single_linkage_classes = np.arange(0, len(distance_matrix))

    # loop until we have 3 clusters
    while len(np.unique(single_linkage_classes)) > 3:

        # find the closest two points
        closest_i = np.argmin(distance_matrix)
        closest_i = np.unravel_index(closest_i, distance_matrix.shape)
        point_one = df.iloc[closest_i[0]][["x", "y"]]
        point_two = df.iloc[closest_i[1]][["x", "y"]]
        # what class are the closest points in?
        point_one_class = single_linkage_classes[closest_i[0]]
        point_two_class = single_linkage_classes[closest_i[1]]

        # collapse the class values of the two closest points into the larger of the two clusters

        # create a mask to match all the current members of each cluster
        x_matches = (single_linkage_classes == point_one_class)
        y_matches = (single_linkage_classes == point_two_class)
        # count how many points are in each cluster
        num_x_matches = x_matches.sum()
        num_y_matches = y_matches.sum()

        # # debugging
        # print(f"Closest points: {point_one['x']}, {point_one['y']} <class {point_one_class}> and {point_two['x']}, {point_two['y']} <class {point_two_class}> with distance {distance_matrix[closest_i]:.4f}")
        # print(f"\tClass {point_one_class} has {num_x_matches} points")
        # print(f"\tClass {point_two_class} has {num_y_matches} points")
        
        # assign the smaller cluster to the larger cluster
        if num_x_matches > num_y_matches:
            single_linkage_classes[y_matches] = point_one_class
        elif num_y_matches > num_x_matches:
            single_linkage_classes[x_matches] = point_two_class
        else:
            single_linkage_classes[y_matches] = point_one_class

        # remove the points so they don't get classified again
        distance_matrix[closest_i[0], :] = np.inf
        distance_matrix[:, closest_i[1]] = np.inf

        print(f"Number of clusters: {len(np.unique(single_linkage_classes))}")

    return single_linkage_classes

In [15]:
single_linkage_classes = single_linkage(df, distance_matrix)

Number of clusters: 311
Number of clusters: 310
Number of clusters: 309
Number of clusters: 308
Number of clusters: 307
Number of clusters: 306
Number of clusters: 305
Number of clusters: 304
Number of clusters: 303
Number of clusters: 302
Number of clusters: 301
Number of clusters: 300
Number of clusters: 299
Number of clusters: 298
Number of clusters: 297
Number of clusters: 296
Number of clusters: 295
Number of clusters: 294
Number of clusters: 293
Number of clusters: 292
Number of clusters: 291
Number of clusters: 290
Number of clusters: 289
Number of clusters: 288
Number of clusters: 287
Number of clusters: 286
Number of clusters: 285
Number of clusters: 284
Number of clusters: 283
Number of clusters: 282
Number of clusters: 281
Number of clusters: 280
Number of clusters: 279
Number of clusters: 278
Number of clusters: 277
Number of clusters: 276
Number of clusters: 275
Number of clusters: 274
Number of clusters: 273
Number of clusters: 272
Number of clusters: 271
Number of cluste

Reassign the class values

In [17]:
def class_value_to_original_class(single_linkage_classes, df):

    # count how many points are in each cluster so we can change the class values to be 1, 2, and 3 instead of the original indices
    for class_value in np.unique(single_linkage_classes):
        # create a mask to match all the current members of this cluster and then count them
        mask = (single_linkage_classes == class_value)
        num_class = mask.sum()
        print(f"Class {class_value}: {num_class} points")

        # match the number of points in each cluster to the original class labels to figure out which cluster is which
        for original_class in df["class"].unique():
            if num_class == len(df[df["class"] == original_class]):
                print(f"\tClass {class_value} is original class {original_class}")
                single_linkage_classes[mask] = original_class
                continue

In [18]:
class_value_to_original_class(single_linkage_classes, df)

Class 100: 106 points
	Class 100 is original class 3
Class 203: 101 points
	Class 203 is original class 1
Class 304: 105 points
	Class 304 is original class 2


## • 3.b) Using the “complete linkage” method, run the hierarchical clustering algorithm on the dataset, and get a 3-cluster result (by cutting the dendrogram at a certain height), and report SSE and RI.

In [ ]:
def complete_linkage(df, distance_matrix):

    # where we'll save the class values
    single_linkage_classes = np.arange(0, len(distance_matrix))

    # loop until we have 3 clusters
    while len(np.unique(single_linkage_classes)) > 3:

        # find the closest two points
        closest_i = np.argmin(distance_matrix)
        closest_i = np.unravel_index(closest_i, distance_matrix.shape)
        point_one = df.iloc[closest_i[0]][["x", "y"]]
        point_two = df.iloc[closest_i[1]][["x", "y"]]
        # what class are the closest points in?
        point_one_class = single_linkage_classes[closest_i[0]]
        point_two_class = single_linkage_classes[closest_i[1]]

        # collapse the class values of the two closest points into the larger of the two clusters

        # create a mask to match all the current members of each cluster
        x_matches = (single_linkage_classes == point_one_class)
        y_matches = (single_linkage_classes == point_two_class)
        # count how many points are in each cluster
        num_x_matches = x_matches.sum()
        num_y_matches = y_matches.sum()

        # # debugging
        # print(f"Closest points: {point_one['x']}, {point_one['y']} <class {point_one_class}> and {point_two['x']}, {point_two['y']} <class {point_two_class}> with distance {distance_matrix[closest_i]:.4f}")
        # print(f"\tClass {point_one_class} has {num_x_matches} points")
        # print(f"\tClass {point_two_class} has {num_y_matches} points")
        
        # assign the smaller cluster to the larger cluster
        if num_x_matches > num_y_matches:
            single_linkage_classes[y_matches] = point_one_class
        elif num_y_matches > num_x_matches:
            single_linkage_classes[x_matches] = point_two_class
        else:
            single_linkage_classes[y_matches] = point_one_class

        # remove the points so they don't get classified again
        distance_matrix[closest_i[0], :] = np.inf
        distance_matrix[:, closest_i[1]] = np.inf

        print(f"Number of clusters: {len(np.unique(single_linkage_classes))}")

    return single_linkage_classes

• 3.c) Using the “average linkage” method, run the hierarchical clustering algorithm on the
dataset, and get a 3-cluster result (by cutting the dendrogram at a certain height), and report
SSE and RI.
• 3.d) Using the “centroid linkage” method, run the hierarchical clustering algorithm on the
dataset, and get a 3-cluster result (by cutting the dendrogram at a certain height), and report
SSE and RI.
• 3.e) Please comment, out of the 4 clustering results (3.a), (3.b), (3.c) and (3.d) which
method gets you the best SSE as well as RI.
• 3.f) Please draw the clustering results (like Figure 1).
